# Transformers

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import seaborn as snss
import pandas as pd
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.grid'] = True
plt.rcParams['axes.axisbelow'] = True
plt.rcParams['figure.dpi'] = 150

In [ ]:
# Check available hardware
import torch
if torch.cuda.is_available():
    print(f'GPU is available, using GPU {torch.cuda.get_device_name(0)}')
    device = torch.device('cuda')
elif torch.mps.is_available():
    print(f'Apple Metal Performance Shaders framework is available, using {torch.device("mps")}')
    device = torch.device('mps')
else:
    print(f'Hardware acceleration not available, using CPU')
    device = torch.device('cpu')

## Implementing scaled dot product attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math

seq = torch.rand(1, 3, 4)
print('Input sequence')
print(seq)
print()

W_q = torch.rand(2, 4)
W_k = torch.rand(2, 4)
W_v = torch.rand(2, 4)

# torch version
print('Output is')
F.scaled_dot_product_attention(
    seq @ W_q.T, seq @ W_k.T, seq @ W_v.T
)

In [ ]:
def scaled_dot_product_attention(query,
                                 key,
                                 value,
                                 W_q,
                                 W_k,
                                 W_v):
    """ Query, Key, Value are batched sequences of vectors of size [B, N, F]
        W_q, W_k, W_v are matrices that map QKV to an embedding space of dim E
    """

result = scaled_dot_product_attention(seq, seq, seq, W_q, W_k, W_v)
print('Output is')
print(result)

## Implementing multi-head self-attention

In [ ]:
mha = nn.MultiheadAttention(embed_dim=4, num_heads=2, batch_first=True)

for param in mha.named_parameters():
    print(param)
    print(param[1].shape)

In [ ]:
outputs, attn_weights = mha(seq, seq, seq)

print('Attention weights')
print(attn_weights)

print('Outputs')
print(outputs)

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, input_dim=4, embed_dim=4, num_heads=2):
        super().__init__()

    def forward(self, seq):

mhsa = MultiHeadSelfAttention()
outputs, attn_weights = mhsa(seq)

print('Attention weights')
print(attn_weights)

print('Outputs')
print(outputs)

In [ ]:
W_q = mha.in_proj_weight[0:4, :].reshape([2, 2, 4])
W_k = mha.in_proj_weight[4:8, :].reshape([2, 2, 4])
W_v = mha.in_proj_weight[8:12, :].reshape([2, 2, 4])
W_o = mha.out_proj.weight

with torch.no_grad():
    mhsa.W_q[:] = W_q
    mhsa.W_k[:] = W_k
    mhsa.W_v[:] = W_v
    mhsa.W_o[:] = W_o

outputs, attn_weights = mhsa(seq)

print('Attention weights')
print(attn_weights)

print('Outputs')
print(outputs)

## PyTorch Transformer

In [ ]:
import torch
import torch.nn as nn

transformer = nn.Transformer(d_model=16, nhead=4, num_encoder_layers=2, batch_first=True)
transformer

## Dataset: Wikitext 2

In [ ]:
!wget -nc -P ~/scr10/wikitext2 https://raw.githubusercontent.com/pytorch/examples/refs/heads/main/word_language_model/data/wikitext-2/train.txt
!wget -nc -P ~/scr10/wikitext2 https://raw.githubusercontent.com/pytorch/examples/refs/heads/main/word_language_model/data/wikitext-2/valid.txt
!wget -nc -P ~/scr10/wikitext2 https://raw.githubusercontent.com/pytorch/examples/refs/heads/main/word_language_model/data/wikitext-2/test.txt

In [ ]:
import os
from collections import Counter

def tokenize(line):
    words = line.split() + ['<eos>']
    return words

def build_vocab(text_path):
    vocab = Counter()
    with open(text_path, 'r') as f:
        for line in f:
            tokens = tokenize(line)
            vocab.update(tokens)

    vocab_enc = {
        '<pad>': 0,
        '<unk>': 1,
    }
    vocab_dec = ['<pad>', '<unk>']
    
    for token, count in sorted(vocab.items(), key=lambda x : x[1], reverse=True):
        if token == '<unk>': 
            continue
        vocab_enc[token] = len(vocab_enc)
        vocab_dec.append(token)
    

    return vocab_enc, vocab_dec

def encode_tokens(tokens, vocab_enc):
    return [ vocab_enc.get(token, vocab_enc['<unk>']) for token in tokens]

vocab_enc, vocab_dec = build_vocab(os.path.expanduser('~/scr10/wikitext2/train.txt'))
print(len(vocab_enc), len(vocab_dec))

In [ ]:
# Really simple dataset
def encode_dataset(text_path, vocab_enc):
    with open(text_path, 'r') as f:
        all_tokens = []
        for line in f:
            tokens = tokenize(line)
            tokens = encode_tokens(tokens, vocab_enc)
            all_tokens.append(torch.LongTensor(tokens))
        all_tokens = torch.concat(all_tokens)

    return all_tokens

def shape_batches(data, batch_size=16):
    num_batches = data.shape[0] // batch_size
    data = data[:num_batches * batch_size]
    data = data.reshape([num_batches, batch_size])
    return data

batch_size = 16
train_data = encode_dataset(os.path.expanduser('~/scr10/wikitext2/train.txt'), vocab_enc)
train_data = shape_batches(train_data, batch_size)
print(train_data.shape)

from torch.utils.data import TensorDataset
train_dataset = TensorDataset(train_data)
val_dataset = TensorDataset(shape_batches(
    encode_dataset(
        os.path.expanduser('~/scr10/wikitext2/valid.txt'), vocab_enc
    ), batch_size
))

print(len(train_dataset), len(val_dataset))

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, num_workers=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, num_workers=2, shuffle=False)

sample,  = next(iter(train_loader))
sample.shape

x = sample[:, :-1]
y = sample[:, 1:]

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, dim, max_len=10000):
        super().__init__()

    def forward(self, x):
        """ X has shape [B, L, E]
        """
        x = x + self.pos_enc[:x.shape[1], :]
        return x

def causal_mask(size, device):
    return torch.log(torch.tril(torch.ones(size, size, device=device)))

class Transformer(nn.Module):
    def __init__(self, vocab_size, embed_size, nhead=2, num_layers=2):
        super().__init__()
        self.embed_size = embed_size
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.positional_encoding = PositionalEncoding(embed_size)
        self.transformer_decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(d_model=embed_size, nhead=nhead, batch_first=True),
            num_layers=num_layers,
            norm=nn.LayerNorm(embed_size)
        )
        self.vocab_decoder = nn.Linear(embed_size, vocab_size)
    
    def forward(self, x):

model = Transformer(len(vocab_enc), 256, nhead=4, num_layers=2)
model = model.to(device)

y_pred = model(x.to(device))

print('Done!')
print(y_pred.shape)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm, trange

def train(model, optimizer, loss_func, train_loader, val_loader, num_epochs=10):
    model = model.to(device)

    train_loss = []
    val_loss = []

    pbar_epoch = trange(num_epochs)
    for ii in pbar_epoch:
        model.train()
        train_loss_ = 0.
        train_correct_ = 0.
        for sample, in tqdm(train_loader, leave=False):
            # Send data to device
            xb = sample[:, :-1].to(device)
            yb = sample[:, 1:].to(device)

            # Make a prediction, compute the loss
            y_pred = model(xb)

            y_pred = y_pred.reshape([-1, y_pred.shape[-1]])
            yb = yb.flatten()
            loss = loss_func(y_pred, yb)

            # Backpropagate, apply gradient updates, zero gradients
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Logging
            train_loss_ += loss.item() * yb.shape[0]
            
        train_loss.append(train_loss_ / len(train_loader.dataset))

        model.eval()
        with torch.no_grad():
            val_loss_ = 0.
            val_correct_ = 0.
            for xb, in tqdm(val_loader, leave=False):
                # Send data to device
                xb = sample[:, :-1].to(device)
                yb = sample[:, 1:].to(device)
    
                # Make a prediction, compute the loss
                y_pred = y_pred.reshape([-1, y_pred.shape[-1]])
                yb = yb.flatten()
                loss = loss_func(y_pred, yb)

                # Logging
                val_loss_ += loss.item() * yb.shape[0]
            val_loss.append(val_loss_ / len(val_loader.dataset))

        pbar_epoch.set_postfix(
            epoch=f"{ii+1:03d} / {num_epochs:03d}",
            train_loss=f"{train_loss[-1]:.3f}",
            val_loss=f"{val_loss[-1]:.3f}")

    return model, (train_loss, val_loss)

# Define a loss function and optimizer
loss = nn.NLLLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model, model_stats = train(model, optimizer, loss, train_loader, val_loader, num_epochs=1)

## Generate new words from a test sample

In [ ]:
test_data = encode_dataset(os.path.expanduser('~/scr10/wikitext2/test.txt'), vocab_enc)
test_data

In [ ]:
def generate_sequence(seq_start, words=16):
    seq = []
    seq_input = seq_start.to(device)
    for ii in range(words):
        output = model(seq_input[None])[0]
        word_probs = output[-1].exp()
        word_idx = torch.multinomial(word_probs, 1)[0]
        seq_input = torch.concat([seq_input, torch.LongTensor([word_idx]).to(device)])
        seq.append(word_idx.item())

    return decode_tokens(seq)

def decode_tokens(tokens, vocab_dec=vocab_dec):
    return [ vocab_dec[ii] for ii in tokens ]

# Start from random token
seq_start = torch.randint(len(vocab_enc), size=[1,], dtype=torch.long).to(device)
print('Sequence start: ', ' '.join(decode_tokens(seq_start)))
print(' '.join(generate_sequence(seq_start)))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, random_split
from tqdm.notebook import tqdm, trange

def train(model, optimizer, loss_func, train_loader, val_loader, num_epochs=10):
    model = model.to(device)

    train_loss = []
    val_loss = []

    pbar_epoch = trange(num_epochs)
    for ii in pbar_epoch:
        model.train()
        train_loss_ = 0.
        train_correct_ = 0.
        for sample, in tqdm(train_loader, leave=False):
            # Send data to device
            xb = sample[:, :-1].to(device)
            yb = sample[:, 1:].to(device)

            # Make a prediction, compute the loss
            y_pred = model(xb)

            y_pred = y_pred.reshape([-1, y_pred.shape[-1]])
            yb = yb.flatten()
            loss = loss_func(y_pred, yb)

            # Backpropagate, apply gradient updates, zero gradients
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # Logging
            train_loss_ += loss.item() * yb.shape[0]
            
        train_loss.append(train_loss_ / len(train_loader.dataset))

        model.eval()
        with torch.no_grad():
            val_loss_ = 0.
            val_correct_ = 0.
            for xb, in tqdm(val_loader, leave=False):
                # Send data to device
                xb = sample[:, :-1].to(device)
                yb = sample[:, 1:].to(device)
    
                # Make a prediction, compute the loss
                y_pred = y_pred.reshape([-1, y_pred.shape[-1]])
                yb = yb.flatten()
                loss = loss_func(y_pred, yb)

                # Logging
                val_loss_ += loss.item() * yb.shape[0]
            val_loss.append(val_loss_ / len(val_loader.dataset))

        pbar_epoch.set_postfix(
            epoch=f"{ii+1:03d} / {num_epochs:03d}",
            train_loss=f"{train_loss[-1]:.3f}",
            val_loss=f"{val_loss[-1]:.3f}")

        # Print out a sample sequence each epoch
        with torch.no_grad():
            seq_start = torch.randint(len(vocab_enc), size=[1,], dtype=torch.long).to(device)
            seq_extend = generate_sequence(seq_start)
            print(' '.join(decode_tokens(seq_start)), ' '.join(seq_extend))

    return model, (train_loss, val_loss)

# Define a loss function and optimizer
loss = nn.NLLLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
model, model_stats = train(model, optimizer, loss, train_loader, val_loader, num_epochs=10)